In [24]:
import pandas as pd
""
df = pd.read_csv('forest_reserve_state.csv')

df.head()

,date,state,area
0,2003-01-01,Johor,356922.0
1,2003-01-01,Kedah,344530.0
2,2003-01-01,Kelantan,629687.0
3,2003-01-01,Melaka,5468.0
4,2003-01-01,Negeri Sembilan,165639.0


In [25]:
# Convert date column to datetime & extract year
df['year'] = pd.to_datetime(df['date']).dt.year

# 🔹 Clean data first
df = df[df['state'] != 'Semenanjung Malaysia']

rename_map = {
    'W.P. Kuala Lumpur': 'Kuala Lumpur',
    'W.P. Labuan': 'Labuan',
    'W.P. Putrajaya': 'Putrajaya'
}
df['state'] = df['state'].replace(rename_map)

# (Optional) Reset index
df = df.reset_index(drop=True)

# 🔹 Now find latest year
latest_year = df['year'].max()

# 🔹 Filter for the latest year
df_latest = df[df['year'] == latest_year]

print("Latest year:", latest_year)
print(df_latest.head())

Latest year: 2021
           date            state      area  year
288  2021-01-01            Johor  334502.0  2021
289  2021-01-01            Kedah  341976.0  2021
290  2021-01-01         Kelantan  629881.0  2021
291  2021-01-01           Melaka    5199.0  2021
292  2021-01-01  Negeri Sembilan  155143.0  2021


In [26]:
df_latest.to_csv("forest_reserve_cleaned.csv", index=False)

In [7]:
import pandas as pd 

df = pd.read_csv("air_pollution.csv", parse_dates=["date"])

print(df.head())

        date pollutant  concentration
0 2017-01-01        CO          0.561
1 2017-02-01        CO          0.530
2 2017-03-01        CO          0.589
3 2017-04-01        CO          0.662
4 2017-05-01        CO            NaN


In [8]:
# Extract year and month
df['year'] = df['date'].dt.year

# make sure concentration is numeric (empty strings -> NaN)
df['concentration'] = pd.to_numeric(df['concentration'], errors='coerce')

df = df[df['year'].isin([2017, 2019, 2021])]

# drop missing concentration values 
df = df.dropna(subset=['concentration'])

# group by pollutant + year, compute average
agg = df.groupby(["pollutant", "year"], as_index=False)["concentration"].mean()

# build hierarchical structure
rows = []
rows.append({'id': 'All', 'parent': '', 'value': ''})   # root -> parent empty cell

for pollutant in sorted(agg['pollutant'].unique()):
    rows.append({'id': pollutant, 'parent': 'All', 'value': ''})
    subset = agg[agg['pollutant'] == pollutant].sort_values('year')
    for _, r in subset.iterrows():
        rows.append({
            'id': f"{pollutant}-{int(r['year'])}",
            'parent': pollutant,
            'value': round(float(r['concentration']), 6)
        })

out = pd.DataFrame(rows, columns=['id','parent','value'])

In [10]:
out.head(30)

,id,parent,value
0,All,,
1,CO,All,
2,CO-2017,CO,0.6459
3,CO-2019,CO,0.650083
4,CO-2021,CO,0.53375
5,NO2,All,
6,NO2-2017,NO2,0.00788
7,NO2-2019,NO2,0.007192
8,NO2-2021,NO2,0.005692
9,O3,All,


In [12]:
out.to_csv("pollutants_cleaned.csv", index=False)
print("Cleaned file saved as pollutants_cleaned.csv")

Cleaned file saved as pollutants_cleaned.csv
